In [ ]:
import numpy as np
from pathlib import Path
import tifffile
from PIL import Image
from typing import List, Dict, Tuple, Union
import json
from collections import defaultdict
from shapely import geometry, ops
from shapely.strtree import STRtree

# Parser anotations

In [1]:
import json
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple


def parse_geojson_segmentation(geojson_path: str) -> Dict[str, List[np.ndarray]]:
    """
    Extrae polígonos por clase desde GeoJSON.
    Retorna: {clase: [array_poligono1, array_poligono2, ...]}
    Cada polígono es array Nx2 de coordenadas (x,y)
    """
    with open(geojson_path) as f:
        data = json.load(f)
    
    annotations = {}
    
    for feature in data.get("features", []):
        props = feature.get("properties", {})
        cls = props.get("name", "unknown")
        if cls in ['Decidua', 'dec', 'dec1', 'dec2', 'Decidual']:
            cls = 'Decidual'
        if cls in ['Labryrinth','Labyrinth', 'lab', 'lab1', 'lab2']:
            cls = 'Labyrinth'
        if cls in ['Junctional Zone', 'jz', 'Junctional']:
            cls = 'Junctional Zone'
            
        geom = feature.get("geometry", {})
        geom_type = geom.get("type")
        coords = geom.get("coordinates", [])
        
        polys = []
        
        if geom_type == "Polygon":
            # coords[0] = exterior, coords[1:] = holes
            if coords:
                exterior = np.array(coords[0])
                polys.append(exterior)
        
        elif geom_type == "MultiPolygon":
            for poly in coords:
                if poly:
                    exterior = np.array(poly[0])
                    polys.append(exterior)
        
        if cls not in annotations:
            annotations[cls] = []
        annotations[cls].extend(polys)
    
    return annotations

# Uniform Sampling (Mismo número de parches por clase)

# Inverse Frequency Sampling 

Le da más peso a clases raras sin descargar completamente la mayoritaria

# Hard Negative Mining 

Selecciona solo los parches más difíciles de la clase mayoritaria, por ej cerca de bordes

In [1]:
import json
import numpy as np
from pathlib import Path
from typing import Dict, List


def parse_geojson(geojson_path: str) -> Dict[str, List[np.ndarray]]:
    """
    Extrae polígonos por clase desde GeoJSON.
    Retorna: {clase: [array(N,2), array(N,2), ...]}
    """
    
    ignore_classes = ['unknown', 'Unknown', 'UNKNOWN', '']
    with open(geojson_path) as f:
        data = json.load(f)
    
    annotations = {}
    
    for feature in data.get("features", []):
        props = feature.get("properties", {})
        cls = props.get("name", "unknown")
        
        if cls in ignore_classes:
            continue
        geom = feature.get("geometry", {})
        geom_type = geom.get("type")
        coords = geom.get("coordinates", [])
        
        polys = []
        
        if geom_type == "Polygon" and coords:
            exterior = np.array(coords[0])
            polys.append(exterior)
            
        elif geom_type == "MultiPolygon":
            for poly in coords:
                if poly:
                    exterior = np.array(poly[0])
                    polys.append(exterior)
        
        if cls not in annotations:
            annotations[cls] = []
        annotations[cls].extend(polys)
    
    return annotations

In [10]:
import numpy as np
import tifffile
from pathlib import Path
from PIL import Image
import random
from collections import defaultdict


class StratifiedTiling:
    def __init__(self, patch_size=1280, overlap=0.2):
        self.patch_size = patch_size
        self.step = int(patch_size * (1 - overlap))
    
    def extract_patches(self, wsi_path, annotations):
        """
        Extrae TODOS los parches válidos de una WSI.
        Retorna: {clase: [parches]}
        """
        img = tifffile.imread(wsi_path)
        h, w = img.shape[:2]
        
        patches_by_class = defaultdict(list)
        
        for cls, polygons in annotations.items():
            for poly in polygons:
                # Bbox del polígono
                px1, py1 = poly.min(axis=0).astype(int)
                px2, py2 = poly.max(axis=0).astype(int)
                px1, py1 = max(0, px1), max(0, py1)
                px2, py2 = min(w, px2), min(h, py2)
                
                # Grid de parches dentro del polígono
                for y in range(py1, py2 - self.patch_size + 1, self.step):
                    for x in range(px1, px2 - self.patch_size + 1, self.step):
                        if x + self.patch_size > w or y + self.patch_size > h:
                            continue
                        
                        # Verificar intersección real
                        if not self._patch_intersects(x, y, poly):
                            continue
                        
                        tile = img[y:y+self.patch_size, x:x+self.patch_size]
                        
                        if self._is_background(tile):
                            continue
                        
                        patches_by_class[cls].append({
                            'coords': (x, y),
                            'tile': tile,
                            'polygon': poly,
                            'wsi': Path(wsi_path).stem,
                            'class': cls
                        })
        
        return dict(patches_by_class), img.shape[:2]
    
    def _patch_intersects(self, x, y, poly):
        """Verifica si parche intersecta polígono."""
        px2, py2 = x + self.patch_size, y + self.patch_size
        poly_x1, poly_y1 = poly.min(axis=0)
        poly_x2, poly_y2 = poly.max(axis=0)
        return max(x, poly_x1) < min(px2, poly_x2) and max(y, poly_y1) < min(py2, poly_y2)
    
    def _is_background(self, tile, threshold=0.95):
        gray = np.mean(tile, axis=2) if tile.ndim == 3 else tile
        return (np.mean(gray > 240) + np.mean(gray < 15)) > threshold

def normalize_class_name(cls: str) -> str:
    """Normaliza nombres de clases con un mapeo centralizado."""
    mapping = {
        # Decidual
        'Decidua': 'Decidual',
        'dec': 'Decidual',
        'dec1': 'Decidual',
        'dec2': 'Decidual',
        'Decidual': 'Decidual',
        
        # Labyrinth
        'Labryinth': 'Labyrinth',  # typo
        'Labyrinth': 'Labyrinth',
        'lab': 'Labyrinth',
        'lab1': 'Labyrinth',
        'lab2': 'Labyrinth',
        
        # Junctional Zone
        'Junctional Zone': 'Junctional Zone',
        'jz': 'Junctional Zone',
        'jz1': 'Junctional Zone',
        'jz2': 'Junctional Zone',
        'Junctional': 'Junctional Zone',
        'Junctional zone': 'Junctional Zone',
    }
    
    # Buscar exacto
    if cls in mapping:
        return mapping[cls]
    
    # Buscar case-insensitive como fallback
    cls_lower = cls.lower().strip()
    for key, value in mapping.items():
        if key.lower() == cls_lower:
            return value
    
    # Si no matchea, devolver original o 'unknown'
    return cls  # o raise ValueError(f"Clase desconocida: {cls}")


# =============================================================================
# TÉCNICA 1: UNIFORM SAMPLING
# Misma cantidad por clase, limitado por la clase más pequeña del dataset
# =============================================================================

def uniform_sampling(all_patches_by_wsi, target_per_class=None):
    """
    Une parches de todas las WSIs y selecciona igual número por clase.
    Si target_per_class es None, usa el mínimo disponible.
    """
    # Acumular por clase
    class_pools = defaultdict(list)
    
    for wsi_patches in all_patches_by_wsi:
        for cls, patches in wsi_patches.items():
            normalized_cls = normalize_class_name(cls)
            
            # Agregar clase normalizada a cada parche también
            for p in patches:
                p['class'] = normalized_cls  # <-- FIX del error anterior
            class_pools[normalized_cls].extend(patches)
    
    # Determinar target
    if target_per_class is None:
        target_per_class = min(len(p) for p in class_pools.values())
    
    print(f"\n[UNIFORM] Target por clase: {target_per_class}")
    for cls, pool in class_pools.items():
        print(f"  {cls}: {len(pool)} disponibles → {target_per_class} seleccionados")
    
    # Seleccionar
    selected = []
    for cls, pool in class_pools.items():
        if len(pool) >= target_per_class:
            selected.extend(random.sample(pool, target_per_class))
        else:
            print(f"  ⚠️ {cls} tiene menos del target ({len(pool)})")
            selected.extend(pool)  # Tomar todos los que hay
    
    random.shuffle(selected)
    return selected


# =============================================================================
# TÉCNICA 2: INVERSE FREQUENCY SAMPLING
# Más parches para clases raras, menos para frecuentes
# =============================================================================

def inverse_frequency_sampling(all_patches_by_wsi, total_patches=5000):
    """
    Distribuye parches inversamente proporcional al área total de cada clase.
    """
    # Calcular áreas totales por clase
    class_areas = defaultdict(float)
    
    for wsi_patches in all_patches_by_wsi:
        for cls, patches in wsi_patches.items():
            normalized_cls = normalize_class_name(cls)
            for p in patches:
                poly = p['polygon']
                area = polygon_area(poly)
                class_areas[normalized_cls] += area
    
    total_area = sum(class_areas.values())
    
    # Pesos inversos
    weights = {cls: total_area / (len(class_areas) * area) 
               for cls, area in class_areas.items()}
    
    # Distribución
    allocation = {}
    remaining = total_patches
    
    for cls, w in sorted(weights.items(), key=lambda x: x[1], reverse=True):
        if cls == list(weights.keys())[-1]:
            allocation[cls] = remaining
        else:
            n = int(total_patches * w / sum(weights.values()))
            allocation[cls] = max(n, 50)  # Mínimo 50
            remaining -= allocation[cls]
    
    print(f"\n[INVERSE FREQUENCY] Total: {total_patches}")
    for cls, area in sorted(class_areas.items(), key=lambda x: x[1], reverse=True):
        pct_area = area / total_area * 100
        n_patches = allocation[cls]
        print(f"  {cls}: {pct_area:.1f}% área → {n_patches} parches")
    
    # Seleccionar
    selected = []
    class_pools = defaultdict(list)
    for wsi_patches in all_patches_by_wsi:
        for cls, patches in wsi_patches.items():
            class_pools[cls].extend(patches)
    
    for cls, n in allocation.items():
        pool = class_pools[cls]
        if len(pool) >= n:
            selected.extend(random.sample(pool, n))
        else:
            print(f"  ⚠️ {cls}: solo {len(pool)} disponibles de {n}")
            selected.extend(pool)
    
    random.shuffle(selected)
    return selected


# =============================================================================
# TÉCNICA 3: HARD NEGATIVE MINING
# Clase mayoritaria: solo parches "difíciles" (bordes, textura)
# Clases minoritarias: todos los disponibles
# =============================================================================

def hard_negative_mining(all_patches_by_wsi, majority_class=None, easy_ratio=0.3):
    """
    Para clase mayoritaria: filtra por dificultad (varianza, bordes).
    Para otras: todos los parches.
    """
    # Identificar clase mayoritaria por área si no se especifica
    if majority_class is None:
        class_areas = defaultdict(float)
        for wsi_patches in all_patches_by_wsi:
            for cls, patches in wsi_patches.items():
                normalized_cls = normalize_class_name(cls)
                for p in patches:
                    class_areas[normalized_cls] += polygon_area(p['polygon'])
        majority_class = max(class_areas, key=class_areas.get)
    
    print(f"\n[HARD NEGATIVE] Mayoritaria: {majority_class} (easy_ratio={easy_ratio})")
    
    selected = []
    
    for wsi_patches in all_patches_by_wsi:
        for cls, patches in wsi_patches.items():
            if cls == majority_class:
                # Calcular dificultad y filtrar
                scored = [(p, difficulty_score(p['tile'])) for p in patches]
                scored.sort(key=lambda x: x[1], reverse=True)
                
                # Quedarse con los más difíciles
                hard_count = int(len(scored) * (1 - easy_ratio))
                hard_count = max(hard_count, min(100, len(scored)))  # Mínimo 100 o todos
                
                selected.extend([p for p, _ in scored[:hard_count]])
                print(f"  {cls}: {len(patches)} → {hard_count} hard negatives")
            else:
                # Clase minoritaria: todos
                selected.extend(patches)
                print(f"  {cls}: {len(patches)} (todos)")
    
    random.shuffle(selected)
    return selected


def polygon_area(poly):
    """Área de polígono (shoelace)."""
    x, y = poly[:, 0], poly[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))


def difficulty_score(tile):
    """Score de dificultad: varianza × gradiente."""
    from scipy import ndimage
    
    gray = np.mean(tile, axis=2) if tile.ndim == 3 else tile.astype(float)
    
    var = np.var(gray)
    dx = ndimage.sobel(gray, axis=0)
    dy = ndimage.sobel(gray, axis=1)
    gradient = np.mean(np.sqrt(dx**2 + dy**2))
    
    return var * gradient

In [11]:
def save_yolo_seg_dataset(patches, output_dir, classes, patch_size):
    """
    Guarda parches en formato YOLO-Seg con identificación de WSI.
    Nombre: {wsi_name}_{idx:04d}.jpg
    """
    output_dir = Path(output_dir)
    (output_dir / "images").mkdir(parents=True, exist_ok=True)
    (output_dir / "labels").mkdir(parents=True, exist_ok=True)
    
    class_to_id = {cls: i for i, cls in enumerate(sorted(classes))}
    
    # Guardar metadata global
    metadata = []
    
    for idx, p in enumerate(patches):
        x, y = p['coords']
        tile = p['tile']
        cls = p['class']
        poly = p['polygon']
        wsi_name = p['wsi']  # Nombre de la WSI de origen
        
        # Nombre: wsi_name + índice
        filename_base = f"{wsi_name}_{idx:04d}"
        
        # Imagen
        img_path = output_dir / "images" / f"{filename_base}.jpg"
        Image.fromarray(tile).save(img_path, quality=95)
        
        # Etiqueta
        label = polygon_to_yolo_seg(poly, x, y, patch_size, class_to_id[cls])
        lbl_path = output_dir / "labels" / f"{filename_base}.txt"
        with open(lbl_path, 'w') as f:
            f.write(label)
        
        # Metadata
        metadata.append({
            'filename': filename_base,
            'wsi': wsi_name,
            'class': cls,
            'class_id': class_to_id[cls],
            'coords': (int(x), int(y)),
            'original_shape': tile.shape[:2]
        })
    
    # Guardar classes.txt
    with open(output_dir / "classes.txt", 'w') as f:
        f.write('\n'.join(sorted(classes)))
    
    # Guardar metadata.json
    import json
    with open(output_dir / "metadata.json", 'w') as f:
        json.dump(metadata, f, indent=2)
    
    # data.yaml
    yaml_content = f"""path: {output_dir.absolute()}
train: images
val: images

nc: {len(classes)}
names: {sorted(classes)}
"""
    with open(output_dir / "data.yaml", 'w') as f:
        f.write(yaml_content)
    
    print(f"\nDataset guardado: {len(patches)} parches en {output_dir}")
    print(f"Metadata: {output_dir}/metadata.json")


def polygon_to_yolo_seg(poly, tile_x, tile_y, patch_size, class_id):
    """Convierte polígono a formato YOLO-Seg."""
    # Recortar al tile
    clipped = clip_polygon(poly, tile_x, tile_y, 
                          tile_x + patch_size, tile_y + patch_size)
    
    if clipped is None or len(clipped) < 3:
        # Fallback: bbox completo
        return f"{class_id} 0.0 0.0 1.0 0.0 1.0 1.0 0.0 1.0"
    
    # Normalizar
    norm = clipped.astype(float)
    norm[:, 0] = (norm[:, 0] - tile_x) / patch_size
    norm[:, 1] = (norm[:, 1] - tile_y) / patch_size
    norm = np.clip(norm, 0, 1)
    
    coords = ' '.join([f"{c:.6f}" for c in norm.flatten()])
    return f"{class_id} {coords}"


def clip_polygon(poly, x1, y1, x2, y2):
    """Sutherland-Hodgman."""
    def inside(p, edge):
        if edge == 0: return p[0] >= x1
        if edge == 1: return p[0] <= x2
        if edge == 2: return p[1] >= y1
        if edge == 3: return p[1] <= y2
    
    def intersect(s, e, edge):
        if edge == 0:
            t = (x1 - s[0]) / (e[0] - s[0])
            return [x1, s[1] + t * (e[1] - s[1])]
        elif edge == 1:
            t = (x2 - s[0]) / (e[0] - s[0])
            return [x2, s[1] + t * (e[1] - s[1])]
        elif edge == 2:
            t = (y1 - s[1]) / (e[1] - s[1])
            return [s[0] + t * (e[0] - s[0]), y1]
        else:
            t = (y2 - s[1]) / (e[1] - s[1])
            return [s[0] + t * (e[0] - s[0]), y2]
    
    output = poly.tolist()
    for edge in range(4):
        input_list = output
        output = []
        if not input_list: break
        s = input_list[-1]
        for e in input_list:
            if inside(e, edge):
                if not inside(s, edge):
                    output.append(intersect(s, e, edge))
                output.append(e)
            elif inside(s, edge):
                output.append(intersect(s, e, edge))
            s = e
    
    return np.array(output) if len(output) >= 3 else None

In [12]:
from pathlib import Path
from sklearn.model_selection import train_test_split





def create_dataset(wsi_dir, geojson_dir, output_dir, 
                   strategy="uniform",  # "uniform", "inverse", "hard_negative"
                   patch_size=1280, 
                   overlap=0.35,
                   target_per_class=None,      # para uniform
                   total_patches=5000,         # para inverse
                   majority_class=None,        # para hard_negative
                   train_ratio=0.8):
    """
    Pipeline completo: 60 WSIs → dataset YOLO-Seg.
    """
    wsi_files = sorted(Path(wsi_dir).glob("*.tif"))
    geojson_files = [Path(geojson_dir) / f"{w.stem}.geojson" for w in wsi_files]
    
    print(f"Procesando {len(wsi_files)} WSIs...")
    
    # Paso 1: Extraer todos los parches de todas las WSIs
    tiler = StratifiedTiling(patch_size=patch_size, overlap=overlap)
    all_patches = []
    
    for wsi_path, geojson_path in zip(wsi_files, geojson_files):
        #if wsi_path.split('/')[-1].split('.')[0] in images_excluded:
        #    continue
        if not geojson_path.exists():
            print(f"⚠️ No encontrado: {geojson_path}")
            continue
        
        annotations = parse_geojson(geojson_path)
        patches_by_class, shape = tiler.extract_patches(wsi_path, annotations)
        
        total = sum(len(p) for p in patches_by_class.values())
        print(f"  {wsi_path.name}: {total} parches | {shape}")
        
        all_patches.append(patches_by_class)
    
    # Paso 2: Aplicar estrategia de muestreo
    if strategy == "uniform":
        selected = uniform_sampling(all_patches, target_per_class)
    elif strategy == "inverse":
        selected = inverse_frequency_sampling(all_patches, total_patches)
    elif strategy == "hard_negative":
        selected = hard_negative_mining(all_patches, majority_class)
    else:
        raise ValueError(f"Estrategia desconocida: {strategy}")
    
    # Paso 3: Split train/val
    train_patches, val_patches = train_test_split(
        selected, train_size=train_ratio, random_state=42, shuffle=True
    )
    
    # Paso 4: Guardar
    all_classes = set()
    for p in selected:
        all_classes.add(p['class'])
    
    save_yolo_seg_dataset(train_patches, f"{output_dir}/train", all_classes, patch_size)
    save_yolo_seg_dataset(val_patches, f"{output_dir}/val", all_classes, patch_size)
    
    # data.yaml final
    with open(f"{output_dir}/dataset.yaml", 'w') as f:
        f.write(f"""path: {Path(output_dir).absolute()}
train: train/images
val: val/images

nc: {len(all_classes)}
names: {sorted(all_classes)}
""")
    
    print(f"\n{'='*50}")
    print(f"DATASET COMPLETO:")
    print(f"  Train: {len(train_patches)}")
    print(f"  Val: {len(val_patches)}")
    print(f"  Total: {len(selected)}")
    print(f"  Estrategia: {strategy}")
    print(f"  Clases: {sorted(all_classes)}")

In [13]:
create_dataset(
    wsi_dir="/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/raw/wsi_images/",
    geojson_dir="/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/raw/annotations_resized/",
    output_dir="/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/YOLO-STRATIFIED-H_NEGATIVE-1280-035ovlp",
    strategy="hard_negative",
    patch_size=1280,
    overlap=0.35,
    target_per_class=1000 )

Procesando 50 WSIs...
  slide_001.tif: 186 parches | (6240, 8686)
  slide_002.tif: 72 parches | (4888, 10126)
  slide_003.tif: 23 parches | (5434, 9768)
  slide_004.tif: 69 parches | (5431, 9049)
  slide_005.tif: 34 parches | (4891, 9406)
  slide_006.tif: 146 parches | (10013, 11569)
  slide_009.tif: 22 parches | (5434, 8688)
  slide_010.tif: 44 parches | (6788, 8332)
  slide_011.tif: 66 parches | (7585, 7965)
  slide_012.tif: 52 parches | (6781, 8684)
  slide_013.tif: 28 parches | (4897, 7246)
  slide_014.tif: 36 parches | (4349, 7963)
  slide_015.tif: 38 parches | (4895, 8686)
  slide_017.tif: 66 parches | (5429, 9768)
  slide_020.tif: 84 parches | (5164, 10131)
  slide_021.tif: 65 parches | (5429, 9406)
  slide_022.tif: 72 parches | (5429, 9408)
  slide_023.tif: 87 parches | (5434, 9047)
  slide_024.tif: 92 parches | (5969, 8334)
  slide_026.tif: 30 parches | (4350, 8687)
  slide_027.tif: 12 parches | (4887, 8688)
  slide_030.tif: 69 parches | (5159, 8689)
  slide_031.tif: 52 parche